In [1]:
!pip install -q sentence-transformers scikit-learn pandas openpyxl

In [2]:
# ============================================================
# ANFIS-LLM Hybrid Loop — REAL implementation of Appendix B.6
# ------------------------------------------------------------
# Appendix B currently says this architecture is "conceptual
# only; no hybrid system was implemented." This script makes
# that no longer true: every step in the B.6 pseudo-code loop
# is implemented with real, running code -- no placeholder
# functions.
#
#   x_t = f_theta(s_t)     -> real MARBERT sentence embedding
#   z_t = PCA_transform    -> real PCA, fit on TRAIN split only
#   mu_t = ANFIS_forward   -> the REAL trained classifier from
#                             v6 (anisotropic Gaussian premise +
#                             learned defuzzification consequent
#                             layer) -- the same model whose
#                             held-out accuracy you already have
#                             real numbers for. Not re-invented,
#                             just reused as a forward pass.
#   c_t = build_context    -> REAL prompt-steering logic: reads
#                             mu_t and writes a system prompt
#                             that reflects the fuzzy membership
#                             (a hard "respond in X" when one
#                             dialect dominates, a soft "blend of
#                             X and Y" when the input is genuinely
#                             ambiguous between two dialects)
#   y_t_hat = LLM_generate -> REAL call to an LLM API (Gemini by
#                             default; OpenAI left as a commented
#                             alternative)
#
# IMPORTANT: the ANFIS half is fit on the TRAIN split only and
# then run on HELD-OUT TEST examples in the demo loop below, so
# the fuzzy layer you see acting on the LLM is the same evaluated
# model, not something re-fit on data it will then "predict."
#
# ============================================================

!pip install -q transformers google-generativeai
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ---- 0. Configure your LLM backend ----
# Get a free Gemini API key at https://aistudio.google.com/apikey
# and paste it below. (Same Google account you used for Opal works.)
GEMINI_API_KEY = "PASTE_YOUR_GEMINI_API_KEY_HERE"  # get one at https://aistudio.google.com/apikey

import google.generativeai as genai
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-3.8-flash")
def llm_generate(user_text, context_prompt):
    """Real call to an LLM, conditioned on the ANFIS-derived context."""
    full_prompt = (
        f"{context_prompt}\n\n"
        f"User message (Arabic): {user_text}\n"
        f"Respond naturally and briefly, in Arabic, following the dialect instruction above."
    )
    response = gemini_model.generate_content(full_prompt)
    return response.text.strip()

# --- Alternative backend: OpenAI (uncomment if you'd rather use this) ---
# from openai import OpenAI
# client = OpenAI(api_key="PASTE_YOUR_OPENAI_API_KEY_HERE")
# def llm_generate(user_text, context_prompt):
#     resp = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[
#             {"role": "system", "content": context_prompt},
#             {"role": "user", "content": user_text},
#         ],
#     )
#     return resp.choices[0].message.content.strip()

# ---- 1. Load data ----
DATA_PATH = "Copy of ARABIC_DIALECTS_ANFIS_GOLD (1)(1) (1).xlsx"
df = pd.read_excel(DATA_PATH, sheet_name="GOLD_Long_UTT_RESP")

# ---- 2. Load MARBERT (the encoder f_theta from Section B.2) ----
model_name = "UBC-NLP/MARBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(DEVICE)
bert_model.eval()

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

@torch.no_grad()
def embed_marbert(texts, batch_size=32):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True, max_length=64, return_tensors="pt"
        ).to(DEVICE)
        out = bert_model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        all_vecs.append(pooled.cpu().numpy())
    return np.vstack(all_vecs)

print("Embedding utterances with MARBERT (this can take a few minutes)...")
utter_emb = embed_marbert(df["utterance"].astype(str).tolist())
print("MARBERT embedding shape:", utter_emb.shape)

# ---- 3. Train/test split (identical seed to v4/v5/v6, for consistency) ----
labels = df["dialect_name"].values
idx = np.arange(len(labels))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, stratify=labels, random_state=RANDOM_SEED
)

# ---- 4. PCA fit on TRAIN ONLY (this is z_t = PCA_transform(x_t) from B.2) ----
pca = PCA(n_components=50, random_state=RANDOM_SEED)
z_train = pca.fit_transform(utter_emb[train_idx])
z_test = pca.transform(utter_emb[test_idx])
y_train = labels[train_idx]
y_test = labels[test_idx]
D = z_train.shape[1]

dialects = ["Levantine", "Egyptian", "Gulf"]

# ---- 5. ANFIS premise layer: anisotropic Gaussian (best-performing from v5/v6) ----
global_var = z_train.var(axis=0)
VAR_FLOOR_FRACTION = 0.05

centroids = {}
variances = {}
for d in dialects:
    mask = y_train == d
    c = z_train[mask].mean(axis=0)
    centroids[d] = c
    per_dim_var = z_train[mask].var(axis=0)
    per_dim_var = np.maximum(per_dim_var, VAR_FLOOR_FRACTION * global_var)
    variances[d] = per_dim_var

def anfis_membership(z, centroids, variances, dialects):
    log_w = []
    for d in dialects:
        var_d = variances[d]
        diff2 = (z - centroids[d]) ** 2
        ll = -0.5 * np.sum(np.log(var_d)) - np.sum(diff2 / (2 * var_d), axis=1)
        log_w.append(ll)
    log_w = np.array(log_w)
    log_w -= log_w.max(axis=0, keepdims=True)
    w = np.exp(log_w)
    w_sum = w.sum(axis=0, keepdims=True)
    mu = w / np.clip(w_sum, 1e-12, None)
    return mu.T

# ---- 6. Defuzzification consequent layer: R fit on TRAIN ONLY (from v6) ----
def one_hot(y, dialects):
    return np.array([[1.0 if lab == d else 0.0 for d in dialects] for lab in y])

mu_train = anfis_membership(z_train, centroids, variances, dialects)
R, *_ = np.linalg.lstsq(mu_train, one_hot(y_train, dialects), rcond=None)

def anfis_forward(text):
    """Full ANFIS_forward(u_t) from B.6 -- real embedding, real PCA,
    real anisotropic Gaussian premise, real learned defuzzification."""
    emb = embed_marbert([text])
    z = pca.transform(emb)
    mu = anfis_membership(z, centroids, variances, dialects)[0]
    crisp = mu @ R
    dominant_idx = int(np.argmax(crisp))
    return mu, dialects[dominant_idx], crisp

# ---- 7. build_context(mu_t, H_{t-1}) -- prompt-steering strategy from B.5.1 ----
def build_context(mu, dialects, ambiguity_margin=0.15):
    order = np.argsort(mu)[::-1]  # descending
    top1, top2 = dialects[order[0]], dialects[order[1]]
    top1_val, top2_val = mu[order[0]], mu[order[1]]

    if (top1_val - top2_val) < ambiguity_margin:
        # Genuinely fuzzy input: the membership vector doesn't cleanly
        # favor one dialect -- tell the LLM to blend, rather than forcing
        # a hard classification decision it doesn't deserve.
        context = (
            f"The user's message shows mixed dialectal features, primarily between "
            f"{top1} Arabic ({top1_val:.2f} membership) and {top2} Arabic ({top2_val:.2f} membership). "
            f"Respond in a way that leans toward {top1} Arabic but stays natural for a "
            f"speaker who code-switches between {top1} and {top2}."
        )
    else:
        context = (
            f"The user's message is classified as {top1} Arabic "
            f"(membership degree {top1_val:.2f}). Respond fluently in {top1} Arabic, "
            f"matching this dialectal register."
        )
    return context

# ---- 8. Full closed-loop demo on REAL held-out test utterances ----
print("\n=== ANFIS -> LLM closed hybrid loop: live demo on held-out test examples ===\n")
rng = np.random.RandomState(RANDOM_SEED)
example_pos = rng.choice(len(test_idx), 5, replace=False)

for i in example_pos:
    row = df.iloc[test_idx[i]]
    s_t = row["utterance"]
    true_dialect = y_test[i]

    mu_t, predicted_dialect, crisp = anfis_forward(s_t)
    c_t = build_context(mu_t, dialects)
    y_hat = llm_generate(s_t, c_t)
    time.sleep(20)  # stay under the free-tier rate limit (5 requests/minute)


    print("-" * 70)
    print("s_t (input):", s_t)
    print("true dialect:", true_dialect, "| ANFIS predicted:", predicted_dialect)
    print("mu_t:", dict(zip(dialects, np.round(mu_t, 3))))
    print("c_t (context sent to LLM):", c_t)
    print("y_t_hat (LLM response):", y_hat)

print("\nDone. Copy this whole block back to Claude, including any errors.")




Using device: cpu


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

...

----------------------------------------------------------------------
s_t (input): كيف حالك؟
true dialect: Gulf | ANFIS predicted: Gulf
mu_t: {'Levantine': np.float32(0.067), 'Egyptian': np.float32(0.218), 'Gulf': np.float32(0.715)}
c_t (context sent to LLM): The user's message is classified as Gulf Arabic (membership degree 0.72). Respond fluently in Gulf Arabic, matching this dialectal register.
y_t_hat (LLM response): بخير الله يسلمك، عساك طيب؟


ERROR:tornado.access:503 POST /v1beta/models/gemini-3.8-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1114.13ms


----------------------------------------------------------------------
s_t (input): لاحظت انو طلعنا منحب نفس نوع الموسيقى؟
true dialect: Levantine | ANFIS predicted: Levantine
mu_t: {'Levantine': np.float32(0.64), 'Egyptian': np.float32(0.059), 'Gulf': np.float32(0.301)}
c_t (context sent to LLM): The user's message is classified as Levantine Arabic (membership degree 0.64). Respond fluently in Levantine Arabic, matching this dialectal register.
y_t_hat (LLM response): إي والله لاحظت! طلع ذوقنا بيشبه بعض كتير. شو أكتر فنان أو غنية عم تسمعها هالأيام؟


ERROR:tornado.access:503 POST /v1beta/models/gemini-3.8-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 785.53ms


----------------------------------------------------------------------
s_t (input): ايه رأيك نروح على السينما
true dialect: Egyptian | ANFIS predicted: Egyptian
mu_t: {'Levantine': np.float32(0.047), 'Egyptian': np.float32(0.89), 'Gulf': np.float32(0.063)}
c_t (context sent to LLM): The user's message is classified as Egyptian Arabic (membership degree 0.89). Respond fluently in Egyptian Arabic, matching this dialectal register.
y_t_hat (LLM response): يلا بينا! فكرة حلوة أوي، تحب نتفرج على فيلم إيه؟

Done. Copy this whole block back to Claude, including any errors.


In [3]:
# ============================================================
# ANFIS layer for Arabic dialect membership inference — v6
# ------------------------------------------------------------
# WHAT'S NEW: a proper DEFUZZIFICATION layer before the decision.
#
# Every version so far (v1-v5) skipped a step. Canonical ANFIS
# (Jang's original architecture) has 5 layers:
#   1) Fuzzification    -- Gaussian membership degree per rule
#   2) Rule firing       -- raw firing strength per rule
#   3) Normalization     -- firing strengths normalized to sum to 1
#                            (this is exactly "mu" in every prior version)
#   4) Defuzzification   -- each rule's CONSEQUENT combined with the
#                            normalized firing strengths -> crisp output
#   5) Decision           -- taken from the crisp output
#
# v1-v5 all jumped straight from step 3 to step 5 -- they argmax'd
# the normalized membership degrees directly, with no consequent
# layer at all. That's a common simplification, but it is not
# actually what "zero-order Sugeno ANFIS" is supposed to do, and
# it's worth fixing properly rather than leaving it implicit.
#
# The fix (standard zero-order Sugeno / Jang's hybrid-learning
# consequent step): each rule d (dialect) gets a constant
# consequent VECTOR r_d in R^3 (one weight per output class). The
# defuzzified crisp output for a sample is the normalized-firing-
# -strength-WEIGHTED SUM of these vectors:
#       y_hat = sum_d  mu_d(z) * r_d
# Stacking the r_d as rows of a 3x3 matrix R, this is just
#       y_hat = mu @ R
# R is fit with ordinary least squares against one-hot training
# labels:
#       R = argmin_R  || one_hot(y_train) - mu_train @ R ||^2
# fit on TRAINING data only (same rule as every other parameter
# in this pipeline) -- so this is still a fully honest, non-
# circular held-out evaluation. It is a genuinely extra learned
# step (a linear layer on top of the fuzzy features), not a
# cosmetic relabeling, so it can really change the result.
#
# This script runs the full comparison on the MARBERT+PCA space
# (since that's the space that matters right now):
#   (a) isotropic premise,  decision WITHOUT defuzzification (= v4's 0.65)
#   (b) isotropic premise,  decision WITH    defuzzification (NEW)
#   (c) anisotropic premise, decision WITHOUT defuzzification (= v5's result)
#   (d) anisotropic premise, decision WITH    defuzzification (NEW)
# all on the identical train/test split, so the four numbers are
# directly comparable.
#
# Same rigor as every version before:
#   - real 80/20 stratified train/test split (identical seed)
#   - PCA fit on training data only
#   - premise parameters (centroids/variances) fit on training data only
#   - consequent matrix R fit on training data only (NEW)
#   - evaluation strictly on the held-out test set
# ============================================================

!pip install -q transformers

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.neighbors import KNeighborsClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("Tip: Runtime > Change runtime type > T4 GPU will make this much faster.")

# ---- 1. Load data ----
DATA_PATH = "/content/Copy of ARABIC_DIALECTS_ANFIS_GOLD (1)(1) (1).xlsx"   # <-- check this matches your Colab Files panel
df = pd.read_excel(DATA_PATH, sheet_name="GOLD_Long_UTT_RESP")

# ---- 2. Load MARBERT ----
model_name = "UBC-NLP/MARBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(DEVICE)
bert_model.eval()


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_marbert(texts, batch_size=32):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True, max_length=64, return_tensors="pt"
        ).to(DEVICE)
        out = bert_model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        all_vecs.append(pooled.cpu().numpy())
        if (i // batch_size) % 10 == 0:
            print(f"  embedded {min(i + len(batch), len(texts))}/{len(texts)}")
    return np.vstack(all_vecs)


print("Embedding utterances with MARBERT (this can take a few minutes)...")
utter_emb = embed_marbert(df["utterance"].astype(str).tolist())
print("MARBERT embedding shape:", utter_emb.shape)

# ---- 3. Train/test split (identical to v4/v5) ----
labels = df["dialect_name"].values
idx = np.arange(len(labels))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, stratify=labels, random_state=RANDOM_SEED
)

# ---- 4. PCA fit on TRAIN ONLY ----
pca = PCA(n_components=50, random_state=RANDOM_SEED)
z_train = pca.fit_transform(utter_emb[train_idx])
z_test = pca.transform(utter_emb[test_idx])
y_train = labels[train_idx]
y_test = labels[test_idx]
D = z_train.shape[1]
print("PCA explained variance (train, MARBERT space):", pca.explained_variance_ratio_.sum())

dialects = ["Levantine", "Egyptian", "Gulf"]


def one_hot(y, dialects):
    return np.array([[1.0 if lab == d else 0.0 for d in dialects] for lab in y])


y_train_oh = one_hot(y_train, dialects)


def fit_defuzz_consequents(mu_train, y_train_oh):
    """Zero-order Sugeno consequent layer: R = argmin_R ||Y - mu_train @ R||^2,
    fit on TRAINING data only. mu_train: (n_train, n_rules). Y: (n_train, n_classes).
    Returns R: (n_rules, n_classes)."""
    R, residuals, rank, sv = np.linalg.lstsq(mu_train, y_train_oh, rcond=None)
    return R


def evaluate(name, mu_train, mu_test, use_defuzz):
    if use_defuzz:
        R = fit_defuzz_consequents(mu_train, y_train_oh)
        crisp_test = mu_test @ R
        pred = np.array(dialects)[crisp_test.argmax(axis=1)]
    else:
        pred = np.array(dialects)[mu_test.argmax(axis=1)]
    acc = accuracy_score(y_test, pred)
    print(f"\n=== {name} ===")
    print("Accuracy:", acc)
    cm = confusion_matrix(y_test, pred, labels=dialects)
    print("Confusion matrix (rows=true, cols=pred), order:", dialects)
    print(cm)
    print(classification_report(y_test, pred, labels=dialects))
    return acc, pred


# ============================================================
# (a)/(b) ISOTROPIC premise functions
# ============================================================
iso_centroids = {}
iso_variances = {}
for d in dialects:
    mask = y_train == d
    c = z_train[mask].mean(axis=0)
    iso_centroids[d] = c
    sq_dists = np.sum((z_train[mask] - c) ** 2, axis=1)
    iso_variances[d] = sq_dists.mean() / D


def anfis_membership_isotropic(z, centroids, variances, dialects):
    D = z.shape[1]
    log_w = np.array(
        [
            -0.5 * D * np.log(variances[d])
            - np.sum((z - centroids[d]) ** 2, axis=1) / (2 * variances[d])
            for d in dialects
        ]
    )
    log_w -= log_w.max(axis=0, keepdims=True)
    w = np.exp(log_w)
    w_sum = w.sum(axis=0, keepdims=True)
    mu = w / np.clip(w_sum, 1e-12, None)
    return mu.T


mu_iso_train = anfis_membership_isotropic(z_train, iso_centroids, iso_variances, dialects)
mu_iso_test = anfis_membership_isotropic(z_test, iso_centroids, iso_variances, dialects)

acc_a, _ = evaluate("(a) Isotropic premise, NO defuzzification [reproduces v4 = 0.65]",
                     mu_iso_train, mu_iso_test, use_defuzz=False)
acc_b, _ = evaluate("(b) Isotropic premise, WITH defuzzification [NEW]",
                     mu_iso_train, mu_iso_test, use_defuzz=True)

# ============================================================
# (c)/(d) ANISOTROPIC (per-dimension) premise functions
# ============================================================
global_var = z_train.var(axis=0)
VAR_FLOOR_FRACTION = 0.05

aniso_centroids = {}
aniso_variances = {}
for d in dialects:
    mask = y_train == d
    c = z_train[mask].mean(axis=0)
    aniso_centroids[d] = c
    per_dim_var = z_train[mask].var(axis=0)
    per_dim_var = np.maximum(per_dim_var, VAR_FLOOR_FRACTION * global_var)
    aniso_variances[d] = per_dim_var


def anfis_membership_anisotropic(z, centroids, variances, dialects):
    log_w = []
    for d in dialects:
        var_d = variances[d]
        diff2 = (z - centroids[d]) ** 2
        ll = -0.5 * np.sum(np.log(var_d)) - np.sum(diff2 / (2 * var_d), axis=1)
        log_w.append(ll)
    log_w = np.array(log_w)
    log_w -= log_w.max(axis=0, keepdims=True)
    w = np.exp(log_w)
    w_sum = w.sum(axis=0, keepdims=True)
    mu = w / np.clip(w_sum, 1e-12, None)
    return mu.T


mu_aniso_train = anfis_membership_anisotropic(z_train, aniso_centroids, aniso_variances, dialects)
mu_aniso_test = anfis_membership_anisotropic(z_test, aniso_centroids, aniso_variances, dialects)

acc_c, _ = evaluate("(c) Anisotropic premise, NO defuzzification [reproduces v5]",
                     mu_aniso_train, mu_aniso_test, use_defuzz=False)
acc_d, pred_d = evaluate("(d) Anisotropic premise, WITH defuzzification [NEW]",
                          mu_aniso_train, mu_aniso_test, use_defuzz=True)

# ---- Baseline: k-NN on the same MARBERT+PCA space, for reference ----
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(z_train, y_train)
knn_acc = accuracy_score(y_test, knn.predict(z_test))

print("\n================ SUMMARY ================")
print(f"(a) Isotropic,  no defuzz   : {acc_a:.4f}")
print(f"(b) Isotropic,  + defuzz    : {acc_b:.4f}")
print(f"(c) Anisotropic, no defuzz  : {acc_c:.4f}")
print(f"(d) Anisotropic, + defuzz   : {acc_d:.4f}")
print(f"k-NN (k=5) baseline          : {knn_acc:.4f}")
print("===========================================")

# ---- Qualitative examples: use the best-performing combo (d) for illustration ----
print("\n=== Example fuzzy membership vectors (anisotropic + defuzzified decision) ===")
rng = np.random.RandomState(RANDOM_SEED)
example_pos = rng.choice(len(z_test), 5, replace=False)
for i in example_pos:
    row = df.iloc[test_idx[i]]
    print(
        row["utterance"],
        "-> true:", y_test[i],
        "| predicted:", pred_d[i],
        "| mu (pre-defuzz):", dict(zip(dialects, np.round(mu_aniso_test[i], 3))),
    )


Using device: cpu
Tip: Runtime > Change runtime type > T4 GPU will make this much faster.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding utterances with MARBERT (this can take a few minutes)...
  embedded 32/2997
  embedded 352/2997
  embedded 672/2997
  embedded 992/2997
  embedded 1312/2997
  embedded 1632/2997
  embedded 1952/2997
  embedded 2272/2997
  embedded 2592/2997
  embedded 2912/2997
MARBERT embedding shape: (2997, 768)
PCA explained variance (train, MARBERT space): 0.7388427

=== (a) Isotropic premise, NO defuzzification [reproduces v4 = 0.65] ===
Accuracy: 0.65
Confusion matrix (rows=true, cols=pred), order: ['Levantine', 'Egyptian', 'Gulf']
[[134  25  41]
 [ 20 125  55]
 [  7  62 131]]
              precision    recall  f1-score   support

   Levantine       0.83      0.67      0.74       200
    Egyptian       0.59      0.62      0.61       200
        Gulf       0.58      0.66      0.61       200

    accuracy                           0.65       600
   macro avg       0.67      0.65      0.65       600
weighted avg       0.67      0.65      0.65       600


=== (b) Isotropic premise, WITH def

In [4]:
# ============================================================
# ENOSH — LIVE demo, backed by the REAL, FINAL trained ANFIS model
# ------------------------------------------------------------
# Updated from the original version to match the final adopted
# model from the paper: MARBERT embeddings + ANISOTROPIC Gaussian
# premise + a learned DEFUZZIFICATION (consequent) layer, instead
# of the original MiniLM + isotropic-only version (46.33% -> now
# 85.33% held-out accuracy). This script is self-contained: it
# re-embeds, refits PCA, and refits the ANFIS layer from scratch,
# so it works even after a fresh Colab runtime restart.
#
# NEW in this version:
#   - Anisotropic Gaussian premise + defuzzification consequent
#     layer (same model as v6 / the paper's final result)
#   - A small floating badge in the corner of the page that shows
#     the detected dialect at a glance, separate from the bigger
#     results panel with the mu bars
#   - Footer accuracy updated to the real final number (85.33%)
#
# HOW TO USE:
#   1. Run this cell as a NEW CELL in your Colab session (it does
#      not depend on any earlier cell -- everything it needs is
#      loaded/re-fit inside this script).
#   2. Wait for MARBERT to load/download and the utterances to be
#      re-embedded (a few minutes on CPU), then ~15-20 seconds for
#      the tunnel to start.
#   3. It will print a line like:
#        YOUR LIVE DEMO IS READY: https://xxxx-xxxx.trycloudflare.com
#   4. Open that link (on the venue's screen, or as a QR code) --
#      that IS the demo. Anyone who opens it can type any Arabic
#      sentence and see REAL, live-computed mu scores from the
#      final trained model.
#   5. KEEP THIS CELL RUNNING for the whole demo. Stopping it, or
#      Colab disconnecting, takes the public link down. If that
#      happens, just re-run the cell -- you'll get a NEW link, so
#      re-open/re-scan it.
#
# A rehearsal tip: run this once before your talk and actually
# open the link on your phone to confirm it works on the venue's
# wifi, since conference wifi can be unpredictable.
# ============================================================

!pip install -q flask transformers
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from flask import Flask, request, jsonify, Response
import threading, subprocess, time, re

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ---- 1. Load data ----
DATA_PATH = "/content/Copy of ARABIC_DIALECTS_ANFIS_GOLD (1)(1) (1).xlsx"   # <-- check this matches your Colab Files panel
df = pd.read_excel(DATA_PATH, sheet_name="GOLD_Long_UTT_RESP")

# ---- 2. Load MARBERT ----
model_name = "UBC-NLP/MARBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(DEVICE)
bert_model.eval()

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

@torch.no_grad()
def embed_marbert(texts, batch_size=32):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True, max_length=64, return_tensors="pt"
        ).to(DEVICE)
        out = bert_model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        all_vecs.append(pooled.cpu().numpy())
    return np.vstack(all_vecs)

print("Embedding utterances with MARBERT (this can take a few minutes)...")
utter_emb = embed_marbert(df["utterance"].astype(str).tolist())
print("MARBERT embedding shape:", utter_emb.shape)

# ---- 3. Train/test split (identical seed to the paper's final model) ----
labels = df["dialect_name"].values
idx = np.arange(len(labels))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, stratify=labels, random_state=RANDOM_SEED
)

# ---- 4. PCA fit on TRAIN ONLY ----
pca = PCA(n_components=50, random_state=RANDOM_SEED)
z_train = pca.fit_transform(utter_emb[train_idx])
y_train = labels[train_idx]
D = z_train.shape[1]

dialects = ["Levantine", "Egyptian", "Gulf"]

# ---- 5. Anisotropic Gaussian premise (the adopted model) ----
global_var = z_train.var(axis=0)
VAR_FLOOR_FRACTION = 0.05

centroids = {}
variances = {}
for d in dialects:
    mask = y_train == d
    c = z_train[mask].mean(axis=0)
    centroids[d] = c
    per_dim_var = z_train[mask].var(axis=0)
    per_dim_var = np.maximum(per_dim_var, VAR_FLOOR_FRACTION * global_var)
    variances[d] = per_dim_var

def anfis_membership(z, centroids, variances, dialects):
    log_w = []
    for d in dialects:
        var_d = variances[d]
        diff2 = (z - centroids[d]) ** 2
        ll = -0.5 * np.sum(np.log(var_d)) - np.sum(diff2 / (2 * var_d), axis=1)
        log_w.append(ll)
    log_w = np.array(log_w)
    log_w -= log_w.max(axis=0, keepdims=True)
    w = np.exp(log_w)
    w_sum = w.sum(axis=0, keepdims=True)
    mu = w / np.clip(w_sum, 1e-12, None)
    return mu.T

# ---- 6. Defuzzification consequent layer: R fit on TRAIN ONLY ----
def one_hot(y, dialects):
    return np.array([[1.0 if lab == d else 0.0 for d in dialects] for lab in y])

mu_train = anfis_membership(z_train, centroids, variances, dialects)
R, *_ = np.linalg.lstsq(mu_train, one_hot(y_train, dialects), rcond=None)

print("Model ready: anisotropic premise + defuzzification, MARBERT embeddings.")

# ============================================================
# Flask app
# ============================================================
app = Flask(__name__)

PAGE_HTML = """<!doctype html>
<html lang="ar">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>ENOSH — Live Demo</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700;900&family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
  :root{
    --bg:#0b1120; --panel:#131c2e; --panel-border:#233149;
    --text:#e8edf7; --muted:#8fa0bd;
    --lev:#60a5fa; --egy:#fbbf24; --gul:#34d399;
    --accent:#a78bfa;
  }
  *{box-sizing:border-box}
  body{
    margin:0; min-height:100vh; background:
      radial-gradient(circle at 20% -10%, #1b2b4d 0%, transparent 55%),
      radial-gradient(circle at 90% 10%, #2a1f4d 0%, transparent 45%),
      var(--bg);
    color:var(--text); font-family:'Inter',sans-serif;
    display:flex; flex-direction:column; align-items:center;
    padding:32px 16px 60px;
  }
  .brand{display:flex; align-items:center; gap:10px; margin-bottom:6px}
  .brand .dot{width:10px;height:10px;border-radius:50%;background:var(--accent);box-shadow:0 0 14px var(--accent)}
  h1{font-family:'Tajawal',sans-serif; font-weight:900; font-size:clamp(28px,5vw,44px); margin:0; letter-spacing:.5px}
  .subtitle{color:var(--muted); font-size:15px; max-width:560px; text-align:center; margin:10px 0 36px; line-height:1.6}
  .card{
    width:100%; max-width:640px; background:var(--panel);
    border:1px solid var(--panel-border); border-radius:20px;
    padding:24px; box-shadow:0 20px 60px rgba(0,0,0,.35);
  }
  .input-row{display:flex; gap:10px}
  textarea{
    flex:1; min-height:64px; resize:vertical;
    background:#0d1526; border:1px solid var(--panel-border); border-radius:12px;
    color:var(--text); font-family:'Tajawal',sans-serif; font-size:19px;
    direction:rtl; text-align:right; padding:14px 16px; line-height:1.6;
  }
  textarea:focus{outline:2px solid var(--accent); outline-offset:1px}
  button{
    background:var(--accent); color:#150f2b; border:none; border-radius:12px;
    padding:0 22px; font-weight:700; font-size:15px; cursor:pointer;
    transition:transform .1s ease, opacity .15s ease;
  }
  button:hover{opacity:.92} button:active{transform:scale(.97)}
  button:disabled{opacity:.5; cursor:default}
  .hint{color:var(--muted); font-size:12.5px; margin-top:10px; direction:rtl; text-align:right}
  .results{margin-top:26px; display:none}
  .results.show{display:block}
  .analyzed-text{
    direction:rtl; text-align:right; font-family:'Tajawal',sans-serif;
    font-size:20px; padding:14px 16px; background:#0d1526;
    border-radius:12px; border:1px solid var(--panel-border); margin-bottom:20px;
  }
  .bar-row{display:flex; align-items:center; gap:12px; margin-bottom:14px}
  .bar-label{width:92px; font-size:13.5px; color:var(--muted); font-weight:600; letter-spacing:.3px}
  .bar-track{flex:1; height:14px; background:#0d1526; border-radius:8px; overflow:hidden; border:1px solid var(--panel-border)}
  .bar-fill{height:100%; border-radius:8px; transition:width .5s cubic-bezier(.22,1,.36,1)}
  .bar-val{width:52px; text-align:left; font-variant-numeric:tabular-nums; font-size:13.5px; color:var(--text); font-weight:600}
  .dominant{
    margin-top:18px; padding:14px 16px; border-radius:12px;
    background:rgba(167,139,250,.12); border:1px solid rgba(167,139,250,.35);
    display:flex; justify-content:space-between; align-items:center;
  }
  .dominant .label{color:var(--muted); font-size:13px}
  .dominant .value{font-size:19px; font-weight:700; color:var(--accent)}
  .sumcheck{margin-top:14px; font-size:12px; color:var(--muted); text-align:center; font-variant-numeric:tabular-nums}
  .sumcheck b{color:#34d399}
  .loading{color:var(--muted); font-size:14px; margin-top:20px; display:none}
  .loading.show{display:block}
  .error{color:#fb7185; font-size:14px; margin-top:16px; display:none}
  .error.show{display:block}
  .footer{color:var(--muted); font-size:12px; margin-top:28px; text-align:center; max-width:520px; line-height:1.6}
  @media (max-width:480px){ .bar-label{width:70px; font-size:12px} }

  /* --- small floating language-indicator badge --- */
  .lang-badge{
    position: fixed; top: 18px; right: 18px;
    display:flex; align-items:center; gap:8px;
    background: var(--panel); border:1px solid var(--panel-border);
    border-radius: 999px; padding: 9px 16px 9px 14px;
    font-size: 13px; font-weight:700; color: var(--text);
    box-shadow: 0 10px 30px rgba(0,0,0,.4);
    opacity: 0; transform: translateY(-10px) scale(.96);
    transition: opacity .35s ease, transform .35s ease, background .35s ease;
    z-index: 1000; pointer-events:none;
  }
  .lang-badge.show{ opacity:1; transform: translateY(0) scale(1); }
  .lang-badge .lang-dot{ width:9px; height:9px; border-radius:50%; flex:none; transition: background .35s ease; box-shadow: 0 0 10px currentColor; }
  .lang-badge .lang-text{ letter-spacing:.2px; white-space:nowrap; }
  @media (max-width:480px){
    .lang-badge{ top:10px; right:10px; padding:7px 12px 7px 10px; font-size:11.5px; }
  }
</style>
</head>
<body>
  <div class="lang-badge" id="langBadge">
    <span class="lang-dot" id="langBadgeDot"></span>
    <span class="lang-text" id="langBadgeText">—</span>
  </div>

  <div class="brand"><span class="dot"></span><span style="color:var(--muted); font-size:13px; letter-spacing:1px; text-transform:uppercase">Live model demo</span></div>
  <h1>ENOSH</h1>
  <p class="subtitle">Type any Arabic sentence below. This calls the real, trained ANFIS layer from the paper — the same model evaluated on the held-out test set — and returns its actual fuzzy membership scores.</p>

  <div class="card">
    <div class="input-row">
      <textarea id="txt" placeholder="اكتب جملة بأي لهجة عربية..."></textarea>
    </div>
    <div style="display:flex; justify-content:flex-end; margin-top:10px">
      <button id="send">Analyze</button>
    </div>
    <div class="hint">اضغط Enter مع Shift لسطر جديد، أو Enter وحدها للإرسال</div>

    <div class="loading" id="loading">Running the real model…</div>
    <div class="error" id="error"></div>

    <div class="results" id="results">
      <div class="analyzed-text" id="analyzedText"></div>

      <div class="bar-row">
        <div class="bar-label">Levantine</div>
        <div class="bar-track"><div class="bar-fill" id="bar-Levantine" style="background:var(--lev); width:0%"></div></div>
        <div class="bar-val" id="val-Levantine">0.00</div>
      </div>
      <div class="bar-row">
        <div class="bar-label">Egyptian</div>
        <div class="bar-track"><div class="bar-fill" id="bar-Egyptian" style="background:var(--egy); width:0%"></div></div>
        <div class="bar-val" id="val-Egyptian">0.00</div>
      </div>
      <div class="bar-row">
        <div class="bar-label">Gulf</div>
        <div class="bar-track"><div class="bar-fill" id="bar-Gulf" style="background:var(--gul); width:0%"></div></div>
        <div class="bar-val" id="val-Gulf">0.00</div>
      </div>

      <div class="dominant">
        <span class="label">Dominant dialect (after defuzzification)</span>
        <span class="value" id="dominantVal">—</span>
      </div>
      <div class="sumcheck">Σμ = <b id="sumVal">—</b> (real fuzzy inference — normalized firing strengths, not a canned number)</div>
    </div>
  </div>

  <div class="footer">Powered by the zero-order Sugeno ANFIS layer trained and evaluated in "Integrating Adaptive Neuro-Fuzzy Inference Systems and Large Language Models for Arabic Dialect Processing" — anisotropic Gaussian premise functions with a learned defuzzification layer, fit on an 80/20 held-out split, 85.33% test accuracy.</div>

<script>
const txt = document.getElementById('txt');
const sendBtn = document.getElementById('send');
const loading = document.getElementById('loading');
const errorBox = document.getElementById('error');
const results = document.getElementById('results');
const langBadge = document.getElementById('langBadge');
const langBadgeDot = document.getElementById('langBadgeDot');
const langBadgeText = document.getElementById('langBadgeText');

const dialectColor = {Levantine:'var(--lev)', Egyptian:'var(--egy)', Gulf:'var(--gul)'};
const dialectLabel = {Levantine:'Levantine Arabic', Egyptian:'Egyptian Arabic', Gulf:'Gulf Arabic'};

async function analyze(){
  const text = txt.value.trim();
  if(!text) return;
  errorBox.classList.remove('show');
  results.classList.remove('show');
  loading.classList.add('show');
  sendBtn.disabled = true;

  try{
    const res = await fetch('/analyze', {
      method:'POST',
      headers:{'Content-Type':'application/json'},
      body: JSON.stringify({text})
    });
    const data = await res.json();
    if(!res.ok) throw new Error(data.error || 'Request failed');

    document.getElementById('analyzedText').textContent = data.text;
    for(const d of ['Levantine','Egyptian','Gulf']){
      const v = data.mu[d] ?? 0;
      document.getElementById('bar-'+d).style.width = (v*100).toFixed(1)+'%';
      document.getElementById('val-'+d).textContent = v.toFixed(3);
    }
    document.getElementById('dominantVal').textContent = data.dominant_dialect;
    document.getElementById('sumVal').textContent = data.sum_check.toFixed(4);
    results.classList.add('show');

    // update the small floating language-indicator badge
    langBadgeDot.style.background = dialectColor[data.dominant_dialect] || 'var(--accent)';
    langBadgeText.textContent = dialectLabel[data.dominant_dialect] || data.dominant_dialect;
    langBadge.classList.add('show');
  }catch(err){
    errorBox.textContent = 'Could not reach the model: ' + err.message;
    errorBox.classList.add('show');
  }finally{
    loading.classList.remove('show');
    sendBtn.disabled = false;
  }
}

sendBtn.addEventListener('click', analyze);
txt.addEventListener('keydown', (e)=>{
  if(e.key === 'Enter' && !e.shiftKey){
    e.preventDefault();
    analyze();
  }
});
</script>
</body>
</html>
"""

@app.route("/")
def home():
    return Response(PAGE_HTML, mimetype="text/html")

@app.route("/analyze", methods=["POST"])
def analyze():
    data = request.get_json(force=True)
    text = (data.get("text") or "").strip()
    if not text:
        return jsonify({"error": "empty text"}), 400

    emb = embed_marbert([text])
    z = pca.transform(emb)
    mu = anfis_membership(z, centroids, variances, dialects)[0]
    mu_dict = {d: float(round(float(m), 4)) for d, m in zip(dialects, mu)}

    # decision uses the DEFUZZIFIED crisp score (mu @ R), matching the
    # final adopted model -- not a raw argmax over mu.
    crisp = mu @ R
    dominant = dialects[int(np.argmax(crisp))]

    return jsonify({
        "text": text,
        "mu": mu_dict,               # raw membership degrees, for the bars
        "dominant_dialect": dominant,  # defuzzified decision, for the badge + label
        "sum_check": round(sum(mu_dict.values()), 4),  # should print ~1.0
    })

def run_flask():
    app.run(host="0.0.0.0", port=5000)

threading.Thread(target=run_flask, daemon=True).start()
time.sleep(2)

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print("Starting tunnel, this can take 10-20 seconds...")
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if m:
        print("\n" + "=" * 60)
        print("YOUR LIVE DEMO IS READY:")
        print(m.group(0))
        print("Open this link (or turn it into a QR code) to use the demo.")
        print("Keep this cell RUNNING for the whole demo.")
        print("=" * 60 + "\n")
        break


Using device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding utterances with MARBERT (this can take a few minutes)...
MARBERT embedding shape: (2997, 768)
Model ready: anisotropic premise + defuzzification, MARBERT embeddings.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Starting tunnel, this can take 10-20 seconds...

YOUR LIVE DEMO IS READY:
https://inkjet-dominant-heart-suspended.trycloudflare.com
Open this link (or turn it into a QR code) to use the demo.
Keep this cell RUNNING for the whole demo.



In [5]:
# ============================================================
# ANFIS mixed-dialect probe — testing Reviewer 2's concern
# ------------------------------------------------------------
# Reviewer 2's point: the dataset only has hard-labeled, single-
# dialect utterances (LEV, EGY, or GUL), never genuinely mixed
# or code-switched examples with a known true blend. So we can't
# fully prove that the ANFIS model's "fuzzy" membership degrees
# actually track real dialect mixing, rather than just being
# leftover uncertainty from an imperfect classifier.
#
# This script builds a SYNTHETIC test of that idea, using data
# we already have, with NO new data collection needed:
#
#   Your corpus has the SAME 999 scenarios written out in all
#   three dialects (LEV/EGY/GUL versions of the same content).
#   For a sample of scenarios, we build a "mixed" sentence by
#   taking the FIRST HALF of the words from one dialect's version
#   and the SECOND HALF of the words from another dialect's
#   version of the SAME scenario, and gluing them together.
#
#   Example (illustrative): if scenario 17's LEV utterance is
#   "شو صاير معك اليوم" and its EGY utterance is "ايه اللي حصلك
#   النهاردة", a LEV+EGY mix might be "شو صاير حصلك النهاردة".
#
#   We KNOW the true composition of each mixed sentence (half
#   dialect A, half dialect B), because we built it ourselves.
#   That gives us a small amount of genuine ground truth for
#   mixing, which the original dataset does not have.
#
# What we check:
#   1. For PURE (unmixed) held-out test sentences, how confident
#      is the trained model (max membership degree) on average?
#   2. For the SYNTHETIC MIXED sentences, is the model LESS
#      confident on average (as it should be, if it is genuinely
#      sensitive to mixing, not just guessing)?
#   3. For the mixed sentences, does the model's TOP-2 dialects
#      by membership actually match the two dialects that were
#      really mixed in (rather than the unrelated third dialect)?
#
# This does NOT fully answer Reviewer 2 -- it's synthetic,
# word-chopped mixing, not natural code-switching by a real
# bilingual speaker, and nobody manually verified these mixed
# sentences read naturally. We say this honestly in the paper.
# But it is a real, run, quantitative check -- not just a claim.
#
# IMPORTANT: this script re-embeds and re-fits the SAME model as
# v6 (identical anisotropic premise + defuzzification), so the
# model being probed here is exactly the one already reported in
# the paper -- nothing new is trained, we're just testing it on
# new synthetic inputs.
# ============================================================

!pip install -q transformers

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ---- 1. Load data ----
DATA_PATH = "Copy of ARABIC_DIALECTS_ANFIS_GOLD (1)(1) (1).xlsx"   # <-- check this matches your Colab Files panel
df = pd.read_excel(DATA_PATH, sheet_name="GOLD_Long_UTT_RESP")

print("\nColumns available in the long-format sheet:")
print(df.columns.tolist())

# ---- 2. Find a scenario-id column so we can match the SAME
#         scenario across its LEV/EGY/GUL versions ----
candidate_names = [
    "scenario_id", "scenario", "id", "ID", "Scenario_ID", "ScenarioID",
    "scenario_num", "scenario_no", "row_id", "index", "Index",
]
scenario_col = None
for c in candidate_names:
    if c in df.columns:
        scenario_col = c
        break

if scenario_col is None:
    print("\n*** Could not auto-detect a scenario-id column. ***")
    print("Please copy the full column list printed above back to Claude")
    print("so we can point the script at the right column name.")
    raise SystemExit(
        "No scenario-id column found -- stopping here, see message above."
    )

print(f"\nUsing '{scenario_col}' as the scenario-id column.")

# Sanity check: each scenario should appear once per dialect (<=3 rows)
counts = df.groupby(scenario_col)["dialect_name"].nunique()
print("Scenarios with all 3 dialects present:", (counts == 3).sum(), "/", len(counts))

# ---- 3. Load MARBERT (identical to v6) ----
model_name = "UBC-NLP/MARBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(DEVICE)
bert_model.eval()


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts_ = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts_


@torch.no_grad()
def embed_marbert(texts, batch_size=32):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True, max_length=64, return_tensors="pt"
        ).to(DEVICE)
        out = bert_model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        all_vecs.append(pooled.cpu().numpy())
    return np.vstack(all_vecs)


print("\nEmbedding utterances with MARBERT (this can take a few minutes)...")
utter_emb = embed_marbert(df["utterance"].astype(str).tolist())

# ---- 4. Identical train/test split, PCA, and ANFIS model as v6 ----
labels = df["dialect_name"].values
idx = np.arange(len(labels))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, stratify=labels, random_state=RANDOM_SEED
)

pca = PCA(n_components=50, random_state=RANDOM_SEED)
z_train = pca.fit_transform(utter_emb[train_idx])
z_test = pca.transform(utter_emb[test_idx])
y_train = labels[train_idx]
y_test = labels[test_idx]

dialects = ["Levantine", "Egyptian", "Gulf"]

global_var = z_train.var(axis=0)
VAR_FLOOR_FRACTION = 0.05
centroids, variances = {}, {}
for d in dialects:
    mask = y_train == d
    c = z_train[mask].mean(axis=0)
    centroids[d] = c
    per_dim_var = z_train[mask].var(axis=0)
    per_dim_var = np.maximum(per_dim_var, VAR_FLOOR_FRACTION * global_var)
    variances[d] = per_dim_var


def anfis_membership(z, centroids, variances, dialects):
    log_w = []
    for d in dialects:
        var_d = variances[d]
        diff2 = (z - centroids[d]) ** 2
        ll = -0.5 * np.sum(np.log(var_d)) - np.sum(diff2 / (2 * var_d), axis=1)
        log_w.append(ll)
    log_w = np.array(log_w)
    log_w -= log_w.max(axis=0, keepdims=True)
    w = np.exp(log_w)
    w_sum = w.sum(axis=0, keepdims=True)
    mu = w / np.clip(w_sum, 1e-12, None)
    return mu.T


def anfis_forward_text(text):
    emb = embed_marbert([text])
    z = pca.transform(emb)
    mu = anfis_membership(z, centroids, variances, dialects)[0]
    return mu


# ---- 5. Baseline: confidence on PURE (unmixed) held-out test sentences ----
mu_test_pure = anfis_membership(z_test, centroids, variances, dialects)
pure_confidence = mu_test_pure.max(axis=1)
print("\n=== Baseline: PURE (unmixed) held-out test sentences ===")
print(f"Average top-membership confidence (max mu): {pure_confidence.mean():.4f}")

# ---- 6. Build synthetic mixed-dialect sentences from TEST-split scenarios only ----
test_df = df.iloc[test_idx].copy()
test_df["dialect_name_"] = test_df["dialect_name"].values

# Only use scenarios where at least 2 dialect versions landed in the TEST split
scenario_groups = test_df.groupby(scenario_col)

pairs = [("Levantine", "Egyptian"), ("Levantine", "Gulf"), ("Egyptian", "Gulf")]
mixed_records = []

for scenario_id, group in scenario_groups:
    available = {row["dialect_name"]: row["utterance"] for _, row in group.iterrows()}
    for d_a, d_b in pairs:
        if d_a in available and d_b in available:
            words_a = str(available[d_a]).split()
            words_b = str(available[d_b]).split()
            if len(words_a) < 2 or len(words_b) < 2:
                continue
            mid_a = len(words_a) // 2
            mid_b = len(words_b) // 2
            # A-first-half + B-second-half
            mixed_1 = " ".join(words_a[:mid_a] + words_b[mid_b:])
            # B-first-half + A-second-half
            mixed_2 = " ".join(words_b[:mid_b] + words_a[mid_a:])
            mixed_records.append((scenario_id, d_a, d_b, mixed_1))
            mixed_records.append((scenario_id, d_a, d_b, mixed_2))

print(f"\nBuilt {len(mixed_records)} synthetic mixed-dialect sentences from held-out test scenarios.")

if len(mixed_records) == 0:
    print("No overlapping test-split scenarios found across dialects -- cannot run the mixed probe.")
    print("Copy this message back to Claude along with the scenario-count sanity check printed above.")
    raise SystemExit("No mixed pairs available.")

# ---- 7. Run the trained model on every synthetic mixed sentence ----
results = []
for scenario_id, d_a, d_b, mixed_text in mixed_records:
    mu = anfis_forward_text(mixed_text)
    mu_by_dialect = dict(zip(dialects, mu))
    top2 = sorted(mu_by_dialect, key=mu_by_dialect.get, reverse=True)[:2]
    correct_top2 = set(top2) == {d_a, d_b}
    results.append({
        "scenario_id": scenario_id,
        "dialect_a": d_a,
        "dialect_b": d_b,
        "mixed_text": mixed_text,
        "mu_LEV": mu_by_dialect["Levantine"],
        "mu_EGY": mu_by_dialect["Egyptian"],
        "mu_GUL": mu_by_dialect["Gulf"],
        "max_mu": mu.max(),
        "top2_correct": correct_top2,
    })

results_df = pd.DataFrame(results)

print("\n=== Synthetic MIXED-dialect sentences ===")
print(f"Average top-membership confidence (max mu): {results_df['max_mu'].mean():.4f}")
print(f"(Compare to {pure_confidence.mean():.4f} for pure, unmixed sentences -- "
      f"lower here would mean the model is genuinely less certain on mixed input.)")

pct_correct_top2 = 100 * results_df["top2_correct"].mean()
print(f"\n% of mixed sentences where the model's TOP-2 dialects by membership "
      f"are exactly the 2 dialects that were actually mixed in: {pct_correct_top2:.1f}%")
print("(If this is well above the ~33% you'd expect from random guessing among the "
      "3 possible pairs, the model is picking up on real signal from both source dialects.)")

print("\n=== 8 example mixed sentences with their membership vectors ===")
for _, row in results_df.sample(min(8, len(results_df)), random_state=RANDOM_SEED).iterrows():
    print("-" * 70)
    print(f"Mixed from: {row['dialect_a']} + {row['dialect_b']}")
    print(f"Mixed text: {row['mixed_text']}")
    print(f"mu: LEV={row['mu_LEV']:.3f}  EGY={row['mu_EGY']:.3f}  GUL={row['mu_GUL']:.3f}")
    print(f"Top-2 dialects match the true 2 sources? {row['top2_correct']}")

print("\nDone. Copy this whole block of output back to Claude, including any errors.")


Using device: cpu

Columns available in the long-format sheet:
['dialect_code', 'dialect_name', 'utterance', 'response']

*** Could not auto-detect a scenario-id column. ***
Please copy the full column list printed above back to Claude
so we can point the script at the right column name.


SystemExit: No scenario-id column found -- stopping here, see message above.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [6]:
# ============================================================
# Statistical robustness checks requested by Reviewer 1 and
# Reviewer 2 -- run this AFTER (or instead of, doesn't matter)
# the mixed-dialect probe script, in the same Colab notebook.
#
# PART A -- Reviewer 1 asked for:
#   "Silhouette scores... or statistical permutation tests"
#   to back up the claim that the 3 dialects are separated in
#   the embedding space, instead of just showing t-SNE pictures.
#
#   This uses the ORIGINAL exploratory embedding model
#   (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2),
#   the SAME one used for the centroid-distance / t-SNE analysis
#   in Sections III-A/III-B of the paper. It does NOT touch the
#   MARBERT classifier -- this is a separate, honest check on the
#   separate exploratory analysis, matching the scope split we
#   already added to the paper.
#
#   We compute:
#     1. The silhouette score for the 3 dialect labels in the
#        50-dim PCA space (ranges -1 to +1; higher = better
#        separated clusters; 0 = no real separation; this is a
#        real, standard clustering-quality statistic, not just
#        a picture).
#     2. A permutation test: we shuffle the dialect labels
#        randomly 1,000 times and recompute the average pairwise
#        centroid distance each time, building a "what if the
#        labels were random" baseline. We then check: is the
#        REAL average centroid distance bigger than almost all of
#        the 1,000 random-shuffle versions? If yes, that's real
#        statistical evidence the separation is not due to chance.
#
# PART B -- Reviewer 2 asked for:
#   "train a baseline dialect classifier... using
#   cross-validation and report macro-F1 and confusion matrices"
#
#   We already did this with ONE train/test split (v6). This part
#   repeats it properly with 5-FOLD CROSS-VALIDATION: the data is
#   split into 5 chunks, we train on 4 chunks and test on the 5th,
#   five times (each chunk gets to be the test set once), and we
#   report the AVERAGE accuracy and macro-F1 across all 5 runs,
#   plus how much they varied. This is the standard way reviewers
#   expect "cross-validation" to be done, instead of one lucky (or
#   unlucky) single split.
#
#   This uses MARBERT (same as v6, same anisotropic premise +
#   defuzzification model) since that's the classifier Section
#   III-C reports on.
# ============================================================

!pip install -q transformers sentence-transformers

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import silhouette_score, accuracy_score, f1_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ---- Load data ----
DATA_PATH = "Copy of ARABIC_DIALECTS_ANFIS_GOLD (1)(1) (1).xlsx"   # <-- check this matches your Colab Files panel
df = pd.read_excel(DATA_PATH, sheet_name="GOLD_Long_UTT_RESP")
labels = df["dialect_name"].values
dialects = ["Levantine", "Egyptian", "Gulf"]

# ============================================================
# PART A -- silhouette score + permutation test on the
#           ORIGINAL exploratory embedding space (MiniLM)
# ============================================================
print("\n" + "=" * 70)
print("PART A: Silhouette score + permutation test (exploratory embedding space)")
print("=" * 70)

print("\nLoading sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...")
minilm = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=DEVICE)

print("Embedding all utterances (this is a small/fast model, should be quick)...")
utter_emb_minilm = minilm.encode(df["utterance"].astype(str).tolist(), batch_size=64, show_progress_bar=True)

pca_explore = PCA(n_components=50, random_state=RANDOM_SEED)
z_explore = pca_explore.fit_transform(utter_emb_minilm)
print("PCA explained variance (exploratory, MiniLM space):", pca_explore.explained_variance_ratio_.sum())

# --- 1. Silhouette score for the 3 dialect labels ---
sil = silhouette_score(z_explore, labels)
print(f"\nSilhouette score for the 3 dialect labels in this 50-dim PCA space: {sil:.4f}")
print("(Ranges from -1 to +1. Above 0 means the dialects form somewhat real clusters;")
print(" closer to 0 means heavy overlap; this is the actual number, not a picture.)")

# --- 2. Permutation test on average pairwise centroid distance ---
def avg_pairwise_centroid_distance(z, labels, dialects):
    centroids = {d: z[labels == d].mean(axis=0) for d in dialects}
    dists = []
    for i in range(len(dialects)):
        for j in range(i + 1, len(dialects)):
            d = np.linalg.norm(centroids[dialects[i]] - centroids[dialects[j]])
            dists.append(d)
    return np.mean(dists)

observed_stat = avg_pairwise_centroid_distance(z_explore, labels, dialects)
print(f"\nObserved average pairwise centroid distance (real labels): {observed_stat:.4f}")

N_PERMUTATIONS = 1000
rng = np.random.RandomState(RANDOM_SEED)
null_stats = np.zeros(N_PERMUTATIONS)
shuffled_labels = labels.copy()
for p in range(N_PERMUTATIONS):
    rng.shuffle(shuffled_labels)
    null_stats[p] = avg_pairwise_centroid_distance(z_explore, shuffled_labels, dialects)

p_value = (np.sum(null_stats >= observed_stat) + 1) / (N_PERMUTATIONS + 1)
print(f"\nPermutation test ({N_PERMUTATIONS} random label shuffles):")
print(f"  Average centroid distance under random shuffling: mean={null_stats.mean():.4f}, "
      f"std={null_stats.std():.4f}, max={null_stats.max():.4f}")
print(f"  p-value (chance of seeing distance >= observed, under random labels): {p_value:.4f}")
print("  (A small p-value, e.g. < 0.01, means the real dialect separation is NOT")
print("   something you'd see just from randomly shuffled labels -- i.e. it's real signal.)")

# ============================================================
# PART B -- 5-fold cross-validation of the ANFIS classifier
#           (MARBERT space, same model as v6)
# ============================================================
print("\n" + "=" * 70)
print("PART B: 5-fold cross-validation of the trained ANFIS classifier (MARBERT space)")
print("=" * 70)

print("\nLoading MARBERT...")
tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/MARBERT")
bert_model = AutoModel.from_pretrained("UBC-NLP/MARBERT").to(DEVICE)
bert_model.eval()


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts_ = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts_


@torch.no_grad()
def embed_marbert(texts, batch_size=32):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
        out = bert_model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        all_vecs.append(pooled.cpu().numpy())
    return np.vstack(all_vecs)


print("Embedding all utterances with MARBERT (this can take a few minutes)...")
utter_emb_marbert = embed_marbert(df["utterance"].astype(str).tolist())


def one_hot(y, dialects):
    return np.array([[1.0 if lab == d else 0.0 for d in dialects] for lab in y])


def anfis_membership_anisotropic(z, centroids, variances, dialects):
    log_w = []
    for d in dialects:
        var_d = variances[d]
        diff2 = (z - centroids[d]) ** 2
        ll = -0.5 * np.sum(np.log(var_d)) - np.sum(diff2 / (2 * var_d), axis=1)
        log_w.append(ll)
    log_w = np.array(log_w)
    log_w -= log_w.max(axis=0, keepdims=True)
    w = np.exp(log_w)
    w_sum = w.sum(axis=0, keepdims=True)
    mu = w / np.clip(w_sum, 1e-12, None)
    return mu.T


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
fold_accuracies = []
fold_macro_f1s = []

for fold_i, (train_idx, test_idx) in enumerate(skf.split(utter_emb_marbert, labels), start=1):
    pca = PCA(n_components=50, random_state=RANDOM_SEED)
    z_train = pca.fit_transform(utter_emb_marbert[train_idx])
    z_test = pca.transform(utter_emb_marbert[test_idx])
    y_train = labels[train_idx]
    y_test = labels[test_idx]

    global_var = z_train.var(axis=0)
    VAR_FLOOR_FRACTION = 0.05
    centroids, variances = {}, {}
    for d in dialects:
        mask = y_train == d
        centroids[d] = z_train[mask].mean(axis=0)
        per_dim_var = z_train[mask].var(axis=0)
        variances[d] = np.maximum(per_dim_var, VAR_FLOOR_FRACTION * global_var)

    mu_train = anfis_membership_anisotropic(z_train, centroids, variances, dialects)
    mu_test = anfis_membership_anisotropic(z_test, centroids, variances, dialects)

    y_train_oh = one_hot(y_train, dialects)
    R, *_ = np.linalg.lstsq(mu_train, y_train_oh, rcond=None)

    crisp_test = mu_test @ R
    pred = np.array(dialects)[crisp_test.argmax(axis=1)]

    acc = accuracy_score(y_test, pred)
    macro_f1 = f1_score(y_test, pred, labels=dialects, average="macro")
    fold_accuracies.append(acc)
    fold_macro_f1s.append(macro_f1)
    print(f"Fold {fold_i}: accuracy = {acc:.4f}, macro-F1 = {macro_f1:.4f}")

fold_accuracies = np.array(fold_accuracies)
fold_macro_f1s = np.array(fold_macro_f1s)

print("\n=== 5-fold cross-validation summary ===")
print(f"Accuracy: mean = {fold_accuracies.mean():.4f}, std = {fold_accuracies.std():.4f}")
print(f"Macro-F1: mean = {fold_macro_f1s.mean():.4f}, std = {fold_macro_f1s.std():.4f}")
print("(Compare mean accuracy here to the single-split result already in the paper: 0.8533)")

print("\nDone. Copy this whole block of output back to Claude, including any errors.")


Using device: cpu

PART A: Silhouette score + permutation test (exploratory embedding space)

Loading sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

PCA explained variance (exploratory, MiniLM space): 0.753891

Silhouette score for the 3 dialect labels in this 50-dim PCA space: -0.0189
(Ranges from -1 to +1. Above 0 means the dialects form somewhat real clusters;
 closer to 0 means heavy overlap; this is the actual number, not a picture.)

Observed average pairwise centroid distance (real labels): 0.3890

Permutation test (1000 random label shuffles):
  Average centroid distance under random shuffling: mean=0.1171, std=0.0113, max=0.1570
  p-value (chance of seeing distance >= observed, under random labels): 0.0010
  (A small p-value, e.g. < 0.01, means the real dialect separation is NOT
   something you'd see just from randomly shuffled labels -- i.e. it's real signal.)

PART B: 5-fold cross-validation of the trained ANFIS classifier (MARBERT space)

Loading MARBERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding all utterances with MARBERT (this can take a few minutes)...
Fold 1: accuracy = 0.8433, macro-F1 = 0.8432
Fold 2: accuracy = 0.8517, macro-F1 = 0.8515
Fold 3: accuracy = 0.8748, macro-F1 = 0.8751
Fold 4: accuracy = 0.8397, macro-F1 = 0.8396
Fold 5: accuracy = 0.8397, macro-F1 = 0.8397

=== 5-fold cross-validation summary ===
Accuracy: mean = 0.8499, std = 0.0132
Macro-F1: mean = 0.8498, std = 0.0134
(Compare mean accuracy here to the single-split result already in the paper: 0.8533)

Done. Copy this whole block of output back to Claude, including any errors.
